### Throughout this notebook advanced Retrieval-Augmented Generation (RAG) techniques have been explored to help with query understanding and document retrieval.

## Topics Covered:


**Hybrid RAG**: It's a FAISS semantic search (with BM25 keyword search) merged RRF technique.


**Query Rewriting RAG**: rewrites user queries to become retrieval-focused and at the same time selects a specific method of retrieval.


**Multi-Query RAG**: this approach is about generating many retrieval points to get better results.


**Query Decomposition RAG**: complex questions are broken down into sub-questions and each sub-question is searched individually.


In [1]:
!pip -q install -U sentence-transformers faiss-cpu google-genai rank-bm25

import os
import re
import json
import numpy as np
import faiss

from getpass import getpass
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from google import genai

GEMINI_API_KEY = getpass("Enter your Gemini API key: ")

os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

client = genai.Client(
    api_key=GEMINI_API_KEY
)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.3/611.3 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.1/259.1 kB 9.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.56.3 which is incompatible.
Enter your Gemini API key: ··········


In [3]:
documents = [

    {
        "id": "doc1",
        "title": "RAG Introduction",
        "text": """
        Retrieval-Augmented Generation, commonly called RAG, combines
        information retrieval with large language models. Instead of relying
        only on the knowledge stored inside an LLM, RAG retrieves relevant
        information from an external knowledge base and provides that
        information to the language model as context. This allows an LLM
        to answer questions using external information.
        """
    },

    {
        "id": "doc2",
        "title": "Vector Databases",
        "text": """
        A vector database stores numerical representations of data called
        embeddings. These embeddings allow the system to perform semantic
        similarity searches. When a user asks a question, the question can
        be converted into an embedding and compared with document embeddings.
        Documents whose embeddings are most similar to the query can then
        be retrieved.
        """
    },

    {
        "id": "doc3",
        "title": "Embeddings",
        "text": """
        Embeddings are numerical vectors that represent the semantic meaning
        of text. Similar pieces of text tend to have embeddings that are close
        to each other in vector space. Embeddings are commonly used for
        semantic search, recommendation systems, clustering, document
        retrieval, and Retrieval-Augmented Generation systems.
        """
    },

    {
        "id": "doc4",
        "title": "RAG Pipeline",
        "text": """
        A typical RAG pipeline consists of document ingestion, text cleaning,
        text splitting, embedding generation, vector storage, retrieval,
        context construction, and language model generation. The retriever
        finds relevant chunks before the language model generates an answer.
        """
    },

    {
        "id": "doc5",
        "title": "Document Chunking",
        "text": """
        Chunking divides large documents into smaller pieces before embedding.
        Good chunking is important because excessively large chunks can contain
        irrelevant information while very small chunks may lose important
        context. Chunk size and overlap should be selected based on the
        document type and retrieval task.
        """
    },

    {
        "id": "doc6",
        "title": "Semantic Search",
        "text": """
        Semantic search retrieves information based on meaning rather than
        relying only on exact keyword matches. A query and documents are
        converted into embeddings, and similarity between the vectors is
        calculated. This allows semantic search to find relevant information
        even when the wording of the query differs from the wording in the
        document.
        """
    },

    {
        "id": "doc7",
        "title": "BM25 Keyword Search",
        "text": """
        BM25 is a lexical information retrieval algorithm. It ranks documents
        based on the occurrence of query terms and their importance within
        the document collection. BM25 is particularly useful for exact
        keywords, technical terminology, product names, identifiers, error
        codes, and other terms where exact matching is important.
        """
    },

    {
        "id": "doc8",
        "title": "Reranking",
        "text": """
        Reranking is a second-stage retrieval process. An initial retriever
        retrieves a larger set of candidate documents and a reranker then
        scores those candidates according to their relevance to the query.
        Reranking can improve precision by moving the most relevant documents
        toward the top of the results.
        """
    },

    {
        "id": "doc9",
        "title": "Hybrid Search",
        "text": """
        Hybrid search combines multiple retrieval methods, commonly semantic
        vector search and lexical keyword search. Vector search is useful for
        semantic meaning while keyword search is useful for exact terms.
        Combining the two approaches can improve retrieval robustness across
        different types of queries.
        """
    },

    {
        "id": "doc10",
        "title": "Authentication Errors",
        "text": """
        Authentication errors can occur when credentials are invalid,
        authentication tokens expire, or authorization policies reject a
        request. Error code ERR-401 commonly indicates an authentication
        failure. Error code ERR-403 commonly indicates that the user is
        authenticated but does not have permission to access a resource.
        """
    },

    {
        "id": "doc11",
        "title": "Product API",
        "text": """
        The Product API provides endpoints for creating, updating, deleting,
        and retrieving product records. Product records contain a product ID,
        name, price, inventory quantity, and category. The API uses JSON for
        request and response bodies.
        """
    },

    {
        "id": "doc12",
        "title": "Query Rewriting",
        "text": """
        Query rewriting transforms a user's original question into a clearer
        and more retrieval-friendly query. Query rewriting can expand missing
        context, resolve vague language, make important concepts explicit,
        remove unnecessary words, and produce terminology that better matches
        the knowledge base.
        """
    },

    {
        "id": "doc13",
        "title": "Query Expansion",
        "text": """
        Query expansion generates multiple alternative search queries from a
        single user query. The alternatives can use synonyms, related concepts,
        different terminology, or different formulations of the same question.
        Multiple searches can improve recall because relevant documents may use
        terminology different from the original user query.
        """
    },

    {
        "id": "doc14",
        "title": "Multi-Query Retrieval",
        "text": """
        Multi-query retrieval uses multiple search queries or perspectives for
        a single information need. Each query retrieves potentially different
        documents. The results are then combined using a ranking or fusion
        method. This approach improves retrieval recall and helps identify
        relevant information that may not be retrieved by a single query.
        """
    },

    {
        "id": "doc15",
        "title": "Query Decomposition",
        "text": """
        Query decomposition breaks a complex user question into smaller
        sub-questions. Each sub-question can be answered independently using
        retrieval. The individual evidence or intermediate answers can then
        be combined to answer the original complex question.
        """
    }
]

print("Number of documents:", len(documents))

Number of documents: 15


In [4]:
def chunk_text(text, chunk_size=80, overlap=20):
    words = text.split()
    step = chunk_size - overlap

    return [
        " ".join(words[i:i + chunk_size])
        for i in range(0, len(words), step)
        if words[i:i + chunk_size]
    ]


chunks = []

for document in documents:
    document_chunks = chunk_text(
        document["text"],
        chunk_size=80,
        overlap=20
    )

    chunks.extend({
        "chunk_id": f'{document["id"]}_chunk_{chunk_number}',
        "document_id": document["id"],
        "title": document["title"],
        "text": chunk
    } for chunk_number, chunk in enumerate(document_chunks))

print("Total chunks:", len(chunks))

Total chunks: 15


In [5]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

embedding_dimension = (
    embedding_model.get_sentence_embedding_dimension()
)

print("Embedding dimension:", embedding_dimension)


chunk_texts = [
    chunk["text"]
    for chunk in chunks
]

chunk_embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

chunk_embeddings = chunk_embeddings.astype("float32")

print("Embedding shape:", chunk_embeddings.shape)


vector_index = faiss.IndexFlatIP(
    embedding_dimension
)

vector_index.add(chunk_embeddings)

print("Vectors:", vector_index.ntotal)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding dimension: 384


/tmp/ipykernel_4073/1300416276.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_model.get_sentence_embedding_dimension()


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding shape: (15, 384)
Vectors: 15


In [6]:
def tokenize(text):

    return re.findall(
        r"\b\w+\b",
        text.lower()
    )


tokenized_chunks = [
    tokenize(chunk["text"])
    for chunk in chunks
]

bm25 = BM25Okapi(
    tokenized_chunks
)

In [7]:
def vector_search(query, top_k=5):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    query_embedding = query_embedding.astype("float32")

    scores, indices = vector_index.search(
        query_embedding,
        top_k
    )

    results = []

    for rank, (score, idx) in enumerate(
        zip(scores[0], indices[0]),
        start=1
    ):

        if idx == -1:
            continue

        result = chunks[idx].copy()

        result["vector_score"] = float(score)
        result["vector_rank"] = rank

        results.append(result)

    return results

In [8]:
def bm25_search(query, top_k=5):

    query_tokens = tokenize(query)

    scores = bm25.get_scores(query_tokens)

    ranked_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for rank, idx in enumerate(
        ranked_indices,
        start=1
    ):

        result = chunks[idx].copy()

        result["bm25_score"] = float(scores[idx])
        result["bm25_rank"] = rank

        results.append(result)

    return results

In [9]:
def reciprocal_rank_fusion(result_lists, k=60):

    fused_results = {}

    for results in result_lists:

        for rank, result in enumerate(
            results,
            start=1
        ):

            chunk_id = result["chunk_id"]

            if chunk_id not in fused_results:

                fused_results[chunk_id] = {
                    "chunk": result,
                    "rrf_score": 0.0
                }

            fused_results[chunk_id]["rrf_score"] += (
                1.0 / (k + rank)
            )

    sorted_results = sorted(
        fused_results.values(),
        key=lambda x: x["rrf_score"],
        reverse=True
    )

    final_results = []

    for item in sorted_results:

        result = item["chunk"].copy()

        result["rrf_score"] = item["rrf_score"]

        final_results.append(result)

    return final_results


def hybrid_search(query, top_k=5, retrieval_k=10):

    vector_results = vector_search(
        query,
        top_k=retrieval_k
    )

    bm25_results = bm25_search(
        query,
        top_k=retrieval_k
    )

    fused_results = reciprocal_rank_fusion(
        [
            vector_results,
            bm25_results
        ]
    )

    return fused_results[:top_k]

In [10]:
def build_context(results):

    context_parts = []

    for i, result in enumerate(results, start=1):

        context_parts.append(
            f"""
SOURCE {i}

Document:
{result["title"]}

Content:
{result["text"]}
"""
        )

    return "\n".join(context_parts)


def generate_answer(question, context):

    prompt = f"""
You are a Retrieval-Augmented Generation assistant.

Answer the user's question using the retrieved context.

QUESTION:
{question}

RETRIEVED CONTEXT:
{context}

RULES:
1. Use the retrieved context as the primary source.
2. Do not invent unsupported information.
3. If the context is insufficient, say so.
4. Give a clear and useful answer.
"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return response.text


def hybrid_rag(question, top_k=5):

    results = hybrid_search(
        question,
        top_k=top_k
    )

    context = build_context(results)

    answer = generate_answer(
        question,
        context
    )

    return {
        "question": question,
        "results": results,
        "context": context,
        "answer": answer
    }

In [11]:
question = """
How does semantic search help a RAG system retrieve relevant information?
"""

result = hybrid_rag(
    question,
    top_k=5
)

print("=" * 70)
print("HYBRID RAG ANSWER")
print("=" * 70)

print(result["answer"])


print("\n" + "=" * 70)
print("RETRIEVED DOCUMENTS")
print("=" * 70)

for i, item in enumerate(
    result["results"],
    start=1
):

    print(
        f"{i}. {item['title']} "
        f"(RRF={item['rrf_score']:.6f})"
    )

HYBRID RAG ANSWER
Semantic search helps a RAG system retrieve relevant information by focusing on the meaning of a query and documents, rather than just exact keyword matches.

Here's how it works within a RAG system:
1.  **Embedding Conversion:** Both the user's query and the documents in the knowledge base are converted into numerical representations called embeddings (Source 1, Source 4).
2.  **Vector Similarity:** These embeddings are stored in a vector database, allowing the system to perform semantic similarity searches. The similarity between the query's embedding and the document embeddings is calculated (Source 1, Source 4).
3.  **Meaning-Based Retrieval:** Documents whose embeddings are most similar to the query's embedding are considered relevant and are then retrieved by RAG's retriever component (Source 3, Source 4).

This process allows the RAG system to find relevant information even when the wording of the query differs from the wording in the document, as long as the u

In [12]:
def rewrite_query(query):

    prompt = f"""
You are a query rewriting component for a Retrieval-Augmented
Generation system.

Rewrite the user's question into a clear,
specific, retrieval-friendly query.

Rules:
1. Preserve the original meaning.
2. Remove unnecessary conversational wording.
3. Make important concepts explicit.
4. Use terminology likely to appear in a technical knowledge base.
5. Return ONLY the rewritten query.

ORIGINAL QUERY:
{query}
"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    rewritten_query = response.text.strip()

    return {
        "original_query": query,
        "rewritten_query": rewritten_query
    }

In [13]:
def classify_query(query):

    router_prompt = f"""
You are a retrieval strategy classifier.

Choose the best retrieval strategy.

Available strategies:

VECTOR:
Use for conceptual, semantic, explanatory,
and meaning-based questions.

BM25:
Use for exact identifiers, error codes,
product names, API names, technical terms,
and exact keyword matching.

HYBRID:
Use when both semantic meaning and exact
keywords are important, or when the query
is complex or comparative.

Return ONLY valid JSON:

{{
    "strategy": "VECTOR" | "BM25" | "HYBRID",
    "reason": "short explanation"
}}

QUERY:
{query}
"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=router_prompt
    )

    text = response.text.strip()

    text = (
        text
        .replace("```json", "")
        .replace("```", "")
        .strip()
    )

    try:

        result = json.loads(text)

        strategy = result.get(
            "strategy",
            "HYBRID"
        ).upper()

        reason = result.get(
            "reason",
            ""
        )

        if strategy not in [
            "VECTOR",
            "BM25",
            "HYBRID"
        ]:
            strategy = "HYBRID"

        return {
            "strategy": strategy,
            "reason": reason
        }

    except Exception:

        return {
            "strategy": "HYBRID",
            "reason": "Router parsing failed; hybrid retrieval selected."
        }

In [14]:
def routed_retrieval(query, top_k=5):

    routing = classify_query(query)

    strategy = routing["strategy"]

    if strategy == "VECTOR":

        results = vector_search(
            query,
            top_k=top_k
        )

    elif strategy == "BM25":

        results = bm25_search(
            query,
            top_k=top_k
        )

    else:

        results = hybrid_search(
            query,
            top_k=top_k
        )

    return {
        "strategy": strategy,
        "reason": routing["reason"],
        "results": results
    }

In [15]:
def query_rewriting_rag(question, top_k=5):

    rewrite_result = rewrite_query(
        question
    )

    rewritten_query = rewrite_result[
        "rewritten_query"
    ]

    retrieval = routed_retrieval(
        rewritten_query,
        top_k=top_k
    )

    context = build_context(
        retrieval["results"]
    )

    prompt = f"""
You are an advanced Retrieval-Augmented Generation assistant.

Answer the original question using the retrieved context.

ORIGINAL QUESTION:
{question}

REWRITTEN RETRIEVAL QUERY:
{rewritten_query}

RETRIEVAL STRATEGY:
{retrieval["strategy"]}

CONTEXT:
{context}

RULES:
1. Answer the original question.
2. Use retrieved context as the primary source.
3. Do not invent unsupported information.
4. If the context is insufficient, say so.
5. Keep the answer clear and useful.
"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return {
        "original_question": question,
        "rewritten_query": rewritten_query,
        "strategy": retrieval["strategy"],
        "routing_reason": retrieval["reason"],
        "results": retrieval["results"],
        "context": context,
        "answer": response.text
    }

In [16]:
question = """
What is it used for in RAG?
"""

result = query_rewriting_rag(
    question,
    top_k=5
)

print("=" * 70)
print("QUERY REWRITING RAG")
print("=" * 70)

print("\nORIGINAL QUESTION:")
print(result["original_question"])

print("\nREWRITTEN QUERY:")
print(result["rewritten_query"])

print("\nSELECTED STRATEGY:")
print(result["strategy"])

print("\nROUTER REASON:")
print(result["routing_reason"])

print("\nANSWER:")
print(result["answer"])

QUERY REWRITING RAG

ORIGINAL QUESTION:

What is it used for in RAG?


REWRITTEN QUERY:
Applications of Retrieval-Augmented Generation (RAG) systems.

SELECTED STRATEGY:
VECTOR

ROUTER REASON:
The query asks for 'Applications' which requires understanding the conceptual use cases and semantic meaning of RAG systems, rather than exact keyword matching or specific identifiers.

ANSWER:
Based on the provided context, Retrieval-Augmented Generation (RAG) is used to enable a large language model (LLM) to answer questions by leveraging external information. Instead of relying solely on the LLM's internal knowledge, RAG retrieves relevant information from an external knowledge base and provides it as context to the language model (Source 1). The retrieved context then helps the language model generate an answer (Source 2).

The context primarily describes *how* RAG works and methods to improve its retrieval capabilities, such as query expansion, multi-query retrieval, and hybrid search (Sourc

In [17]:
def generate_multi_queries(
    question,
    num_queries=3
):

    prompt = f"""
You are a multi-query generation component
for a Retrieval-Augmented Generation system.

Generate {num_queries} different search queries
for the same information need.

Each query should approach the question
from a different useful perspective.

The queries should:
- preserve the original information need
- use different terminology where useful
- improve retrieval recall
- be suitable for semantic and keyword retrieval

Return ONLY valid JSON:

{{
    "queries": [
        "query 1",
        "query 2",
        "query 3"
    ]
}}

USER QUESTION:
{question}
"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    text = response.text.strip()

    text = (
        text
        .replace("```json", "")
        .replace("```", "")
        .strip()
    )

    try:

        result = json.loads(text)

        queries = result.get(
            "queries",
            []
        )

        if not queries:
            queries = [question]

        return queries[:num_queries]

    except Exception:

        return [question]

In [18]:
def multi_query_retrieval(
    queries,
    retrieval_k=5,
    final_top_k=5
):

    all_results = []

    for query in queries:

        results = hybrid_search(
            query,
            top_k=retrieval_k
        )

        all_results.append(results)

    fused_results = reciprocal_rank_fusion(
        all_results
    )

    return fused_results[:final_top_k]

In [19]:
def multi_query_rag(
    question,
    num_queries=3,
    final_top_k=5
):

    queries = generate_multi_queries(
        question,
        num_queries=num_queries
    )

    results = multi_query_retrieval(
        queries,
        retrieval_k=5,
        final_top_k=final_top_k
    )

    context = build_context(
        results
    )

    prompt = f"""
You are an advanced Retrieval-Augmented Generation assistant.

Answer the original question using the combined
retrieved evidence.

ORIGINAL QUESTION:
{question}

SEARCH QUERIES:
{chr(10).join(
    f"{i+1}. {q}"
    for i, q in enumerate(queries)
)}

RETRIEVED CONTEXT:
{context}

RULES:
1. Answer the original question.
2. Use the retrieved context as the primary source.
3. Do not invent unsupported information.
4. If the context is insufficient, say so.
5. Keep the answer clear and useful.
"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return {
        "question": question,
        "queries": queries,
        "results": results,
        "context": context,
        "answer": response.text
    }

In [20]:
question = """
How can RAG retrieve information accurately when
the user's wording is different from the documents?
"""

result = multi_query_rag(
    question,
    num_queries=3,
    final_top_k=5
)

print("=" * 70)
print("MULTI-QUERY RAG")
print("=" * 70)

print("\nORIGINAL QUESTION:")
print(result["question"])

print("\nGENERATED SEARCH QUERIES:")

for i, query in enumerate(
    result["queries"],
    start=1
):
    print(f"{i}. {query}")

print("\nANSWER:")
print(result["answer"])

print("\nRETRIEVED DOCUMENTS:")

for i, item in enumerate(
    result["results"],
    start=1
):

    print(
        f"{i}. {item['title']} "
        f"(RRF={item['rrf_score']:.6f})"
    )

MULTI-QUERY RAG

ORIGINAL QUESTION:

How can RAG retrieve information accurately when
the user's wording is different from the documents?


GENERATED SEARCH QUERIES:
1. RAG strategies for handling vocabulary mismatch between user queries and retrieved documents
2. Mechanisms for RAG to achieve accurate retrieval with diverse user query wording
3. How RAG systems improve retrieval recall when user queries are paraphrased or semantically similar

ANSWER:
RAG can retrieve information accurately when the user's wording differs from the documents through several mechanisms:

1.  **Semantic Search:** This method retrieves information based on the *meaning* of the query rather than relying solely on exact keyword matches. Both the user's query and the documents are converted into embeddings (vector representations), and the system calculates the similarity between these vectors. This allows for finding relevant information even when the specific words used in the query are different from thos

In [21]:
def decompose_query(
    question,
    max_subquestions=4
):

    prompt = f"""
You are a query decomposition component
for a Retrieval-Augmented Generation system.

Break the complex user question into
smaller independent sub-questions.

Each sub-question should:
- address one important part of the question
- be understandable independently
- be suitable for information retrieval
- collectively cover the original question

Return ONLY valid JSON:

{{
    "sub_questions": [
        "sub-question 1",
        "sub-question 2",
        "sub-question 3"
    ]
}}

ORIGINAL QUESTION:
{question}
"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    text = response.text.strip()

    text = (
        text
        .replace("```json", "")
        .replace("```", "")
        .strip()
    )

    try:

        result = json.loads(text)

        sub_questions = result.get(
            "sub_questions",
            []
        )

        return sub_questions[:max_subquestions]

    except Exception:

        return [question]

In [22]:
def retrieve_decomposed_queries(
    sub_questions,
    top_k=3
):

    results = []

    for sub_question in sub_questions:

        retrieved = hybrid_search(
            sub_question,
            top_k=top_k
        )

        results.append({
            "sub_question": sub_question,
            "results": retrieved
        })

    return results

In [23]:
def fuse_decomposed_evidence(
    subquestion_results,
    final_top_k=8
):

    result_lists = [
        item["results"]
        for item in subquestion_results
    ]

    fused_results = reciprocal_rank_fusion(
        result_lists
    )

    return fused_results[:final_top_k]

In [24]:
def query_decomposition_rag(
    question,
    max_subquestions=4,
    final_top_k=8
):

    # Step 1: Rewrite
    rewrite_result = rewrite_query(
        question
    )

    rewritten_query = rewrite_result[
        "rewritten_query"
    ]

    # Step 2: Decompose
    sub_questions = decompose_query(
        rewritten_query,
        max_subquestions=max_subquestions
    )

    # Step 3: Retrieve independently
    subquestion_results = (
        retrieve_decomposed_queries(
            sub_questions,
            top_k=3
        )
    )

    # Step 4: Fuse evidence
    fused_results = (
        fuse_decomposed_evidence(
            subquestion_results,
            final_top_k=final_top_k
        )
    )

    # Step 5: Build context
    context = build_context(
        fused_results
    )

    # Step 6: Generate final answer
    prompt = f"""
You are an advanced Retrieval-Augmented Generation assistant.

Answer the original complex question using
the evidence retrieved for its sub-questions.

ORIGINAL QUESTION:
{question}

REWRITTEN QUESTION:
{rewritten_query}

SUB-QUESTIONS:
{chr(10).join(
    f"{i+1}. {q}"
    for i, q in enumerate(sub_questions)
)}

EVIDENCE:
{context}

RULES:
1. Answer the original question.
2. Combine the evidence logically.
3. Use retrieved evidence as the primary source.
4. Do not invent unsupported information.
5. If the evidence is insufficient, say so.
"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return {
        "original_question": question,
        "rewritten_query": rewritten_query,
        "sub_questions": sub_questions,
        "subquestion_results": subquestion_results,
        "fused_results": fused_results,
        "context": context,
        "answer": response.text
    }

In [25]:
question = """
How does RAG use embeddings and hybrid search
to retrieve relevant information before generating
an answer?
"""

result = query_decomposition_rag(
    question,
    max_subquestions=4,
    final_top_k=8
)

print("=" * 70)
print("QUERY DECOMPOSITION RAG")
print("=" * 70)

print("\nORIGINAL QUESTION:")
print(result["original_question"])

print("\nREWRITTEN QUESTION:")
print(result["rewritten_query"])

print("\nSUB-QUESTIONS:")

for i, question in enumerate(
    result["sub_questions"],
    start=1
):

    print(f"{i}. {question}")

print("\nFINAL ANSWER:")
print(result["answer"])

QUERY DECOMPOSITION RAG

ORIGINAL QUESTION:

How does RAG use embeddings and hybrid search
to retrieve relevant information before generating
an answer?


REWRITTEN QUESTION:
Explain the process by which Retrieval-Augmented Generation (RAG) leverages vector embeddings and hybrid search for information retrieval.

SUB-QUESTIONS:
1. How are vector embeddings utilized for information retrieval in Retrieval-Augmented Generation (RAG)?
2. What is the role and mechanism of hybrid search within the RAG process for information retrieval?
3. How do vector embeddings and hybrid search work together to enhance the information retrieval process in RAG?

FINAL ANSWER:
Retrieval-Augmented Generation (RAG) utilizes vector embeddings and hybrid search to efficiently retrieve relevant information before generating an answer. This process allows a large language model (LLM) to leverage external knowledge beyond its internal training data (Source 1).

**Vector Embeddings for Information Retrieval in RAG: